In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
eval_set = pd.read_csv("../eval/eval_set.csv")

In [3]:
df = pd.read_csv(
    "hf://datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/"
    "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
)

In [4]:
# Remove duplicate intent-response pairs
clean_df = df.drop_duplicates(
    subset=["intent", "response"]
).copy()

# Recreate KB
kb_data = []

for intent in clean_df["intent"].unique():

    intent_data = clean_df[
        clean_df["intent"] == intent
    ]

    responses = intent_data["response"].sample(
        n=min(10, len(intent_data)),
        random_state=42
    )

    kb_data.append({
        "intent": intent,
        "category": intent_data["category"].iloc[0],
        "article_text": " ".join(responses)
    })

kb = pd.DataFrame(kb_data)

# Recreate article IDs
kb.insert(
    0,
    "article_id",
    ["KB_" + str(i).zfill(2) for i in range(1, len(kb) + 1)]
)

# Save the KB
kb.to_csv("kb.csv", index=False)

print("kb.csv recreated successfully!")
print("Shape:", kb.shape)

kb.csv recreated successfully!
Shape: (27, 4)


In [5]:
kb = pd.read_csv("kb.csv")

In [6]:
df = pd.read_csv(
    "hf://datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/"
    "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
)

In [7]:
kb

,article_id,intent,category,article_text
0,KB_01,cancel_order,ORDER,I understand the financial constraints you are...
1,KB_02,change_order,ORDER,Thanks for dropping us a line to us with your ...
2,KB_03,change_shipping_address,SHIPPING,We apologize for the inconvenience you are fac...
3,KB_04,check_cancellation_fee,CANCEL,"Certainly! To view the early exit penalty, you..."
4,KB_05,check_invoice,INVOICE,I've understood you're looking for quick assis...
5,KB_06,check_payment_methods,PAYMENT,I'll make it happen! I would be happy to assis...
6,KB_07,check_refund_policy,REFUND,Of course! I'm here to assist you in understan...
7,KB_08,complaint,FEEDBACK,I'm sorry to hear that you need to make a cons...
8,KB_09,contact_customer_service,CONTACT,Honored to assist! I'm clued in that you would...
9,KB_10,contact_human_agent,CONTACT,Thank you for trusting us! I'm fully aware of ...


In [8]:
eval_set

,query_id,instruction,intent,category
0,Q01,I cannot afford purchase {{Order Number}},cancel_order,ORDER
1,Q02,modify order {{Order Number}},change_order,ORDER
2,Q03,where do I correct my address?,change_shipping_address,SHIPPING
3,Q04,where can I see the temrination fees?,check_cancellation_fee,CANCEL
4,Q05,i dont know what i need to do to find ibll #12588,check_invoice,INVOICE
5,Q06,show me your payment options,check_payment_methods,PAYMENT
6,Q07,can you show me in what cases can I ask to be ...,check_refund_policy,REFUND
7,Q08,help making a customer reclamation against you...,complaint,FEEDBACK
8,Q09,can I contact customer assistance?,contact_customer_service,CONTACT
9,Q10,what do i need to do to talk to someone,contact_human_agent,CONTACT


In [9]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

kb_tfidf = vectorizer.fit_transform(
    kb["article_text"]
)

print("KB articles:", len(kb))
print("TF-IDF matrix:", kb_tfidf.shape)

KB articles: 27
TF-IDF matrix: (27, 8344)


In [10]:
vectorizer

,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\

In [11]:
def search_tfidf(query, top_k=5):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        kb_tfidf
    ).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = kb.iloc[top_indices][
        ["article_id", "intent", "category"]
    ].copy()

    results["score"] = scores[top_indices]

    return results.reset_index(drop=True)

In [12]:
print(kb.columns.tolist())
print(kb.shape)

['article_id', 'intent', 'category', 'article_text']
(27, 4)


In [13]:
kb

,article_id,intent,category,article_text
0,KB_01,cancel_order,ORDER,I understand the financial constraints you are...
1,KB_02,change_order,ORDER,Thanks for dropping us a line to us with your ...
2,KB_03,change_shipping_address,SHIPPING,We apologize for the inconvenience you are fac...
3,KB_04,check_cancellation_fee,CANCEL,"Certainly! To view the early exit penalty, you..."
4,KB_05,check_invoice,INVOICE,I've understood you're looking for quick assis...
5,KB_06,check_payment_methods,PAYMENT,I'll make it happen! I would be happy to assis...
6,KB_07,check_refund_policy,REFUND,Of course! I'm here to assist you in understan...
7,KB_08,complaint,FEEDBACK,I'm sorry to hear that you need to make a cons...
8,KB_09,contact_customer_service,CONTACT,Honored to assist! I'm clued in that you would...
9,KB_10,contact_human_agent,CONTACT,Thank you for trusting us! I'm fully aware of ...


In [14]:
query = eval_set.iloc[0]["instruction"]

print("QUERY:", query)
print("EXPECTED:", eval_set.iloc[0]["intent"])

search_tfidf(query, top_k=5)

QUERY: I cannot afford purchase {{Order Number}}
EXPECTED: cancel_order


,article_id,intent,category,score
0,KB_02,change_order,ORDER,0.492449
1,KB_26,track_order,ORDER,0.446029
2,KB_01,cancel_order,ORDER,0.426373
3,KB_14,delivery_period,DELIVERY,0.192102
4,KB_20,place_order,ORDER,0.127014


In [15]:
def evaluate_tfidf(eval_set, top_k=5):

    results = []

    for _, row in eval_set.iterrows():

        retrieved = search_tfidf(
            row["instruction"],
            top_k=top_k
        )

        predicted_intents = retrieved["intent"].tolist()

        results.append({
            "query_id": row["query_id"],
            "expected_intent": row["intent"],
            "top_intent": predicted_intents[0],
            "top_k_intents": predicted_intents,
            "correct_at_1": row["intent"] == predicted_intents[0],
            "correct_at_5": row["intent"] in predicted_intents
        })

    return pd.DataFrame(results)

In [16]:
tfidf_results = evaluate_tfidf(
    eval_set,
    top_k=5
)

tfidf_results.head()

,query_id,expected_intent,top_intent,top_k_intents,correct_at_1,correct_at_5
0,Q01,cancel_order,change_order,"[change_order, track_order, cancel_order, deli...",False,True
1,Q02,change_order,track_order,"[track_order, cancel_order, change_order, deli...",False,True
2,Q03,change_shipping_address,change_shipping_address,"[change_shipping_address, set_up_shipping_addr...",True,True
3,Q04,check_cancellation_fee,check_cancellation_fee,"[check_cancellation_fee, track_order, track_re...",True,True
4,Q05,check_invoice,track_order,"[track_order, check_invoice, check_payment_met...",False,True


In [17]:
top1_accuracy = tfidf_results["correct_at_1"].mean()
top5_recall = tfidf_results["correct_at_5"].mean()

print(f"TF-IDF Top-1 Accuracy: {top1_accuracy:.3f}")
print(f"TF-IDF Recall@5:       {top5_recall:.3f}")

TF-IDF Top-1 Accuracy: 0.700
TF-IDF Recall@5:       1.000


In [18]:
all_queries_tfidf = vectorizer.transform(
    df["instruction"]
)

print(all_queries_tfidf.shape)

(26872, 8344)


In [19]:
similarities = cosine_similarity(
    all_queries_tfidf,
    kb_tfidf
)

print(similarities.shape)

(26872, 27)


In [20]:
top_indices = similarities.argmax(axis=1)

predicted_intents = kb.iloc[
    top_indices
]["intent"].values

In [21]:
predicted_intents

<ArrowStringArray>
[        'track_order',         'track_order',         'track_order',
        'cancel_order',        'cancel_order',         'track_order',
         'track_order',        'change_order',        'change_order',
        'change_order',
 ...
        'track_refund', 'check_refund_policy',        'track_refund',
          'get_refund',        'track_refund',          'get_refund',
        'track_refund',        'track_refund',        'track_refund',
        'track_refund']
Length: 26872, dtype: str

In [22]:
accuracy = (
    predicted_intents == df["intent"].values
).mean()

print(f"TF-IDF Top-1 Accuracy: {accuracy:.4f}")

TF-IDF Top-1 Accuracy: 0.6997


In [23]:
# Get top 5 articles for each ticket
top5 = similarities.argsort(axis=1)[:, -5:]

# Check if the correct intent is in the top 5
correct = 0

for i in range(len(df)):
    predicted_intents = kb.iloc[top5[i]]["intent"].values
    
    if df.iloc[i]["intent"] in predicted_intents:
        correct += 1

recall_at_5 = correct / len(df)

print(f"TF-IDF Recall@5: {recall_at_5:.4f}")

TF-IDF Recall@5: 0.9672


Sentence transformer


In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
kb_embeddings = model.encode(
    kb["article_text"].tolist(),
    show_progress_bar=True
)

print(kb_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(27, 384)


In [26]:
ticket_embeddings = model.encode(
    df["instruction"].tolist(),
    show_progress_bar=True
)

print(ticket_embeddings.shape)

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

(26872, 384)


In [27]:
embedding_similarities = cosine_similarity(
    ticket_embeddings,
    kb_embeddings
)

print(embedding_similarities.shape)

(26872, 27)


In [28]:
top_indices = embedding_similarities.argmax(axis=1)

predicted_intents = kb.iloc[
    top_indices
]["intent"].values

accuracy = (
    predicted_intents == df["intent"].values
).mean()

print(f"Embedding Top-1 Accuracy: {accuracy:.4f}")

Embedding Top-1 Accuracy: 0.8078


In [29]:
top5 = embedding_similarities.argsort(axis=1)[:, -5:]

correct = 0

for i in range(len(df)):
    predicted_intents = kb.iloc[top5[i]]["intent"].values
    
    if df.iloc[i]["intent"] in predicted_intents:
        correct += 1

recall_at_5 = correct / len(df)

print(f"Embedding Recall@5: {recall_at_5:.4f}")

Embedding Recall@5: 0.9903


In [30]:
# Get predictions from both models
tfidf_top1 = similarities.argmax(axis=1)
embedding_top1 = embedding_similarities.argmax(axis=1)

tfidf_pred = kb.iloc[tfidf_top1]["intent"].values
embedding_pred = kb.iloc[embedding_top1]["intent"].values

# Find cases where TF-IDF is wrong but embeddings are correct
better_cases = df[
    (tfidf_pred != df["intent"].values) &
    (embedding_pred == df["intent"].values)
].copy()

print("Cases where embeddings fixed TF-IDF:")
print(len(better_cases))

Cases where embeddings fixed TF-IDF:
5816


In [31]:
for i, row in better_cases.head(10).iterrows():
    print("\nTicket:", row["instruction"])
    print("Expected:", row["intent"])
    print("TF-IDF:", tfidf_pred[i])
    print("Embedding:", embedding_pred[i])


Ticket: question about cancelling order {{Order Number}}
Expected: cancel_order
TF-IDF: track_order
Embedding: cancel_order

Ticket: i have a question about cancelling oorder {{Order Number}}
Expected: cancel_order
TF-IDF: track_order
Embedding: cancel_order

Ticket: i need help cancelling puchase {{Order Number}}
Expected: cancel_order
TF-IDF: track_order
Embedding: cancel_order

Ticket: can you help me cancel order {{Order Number}}?
Expected: cancel_order
TF-IDF: track_order
Embedding: cancel_order

Ticket: I can no longer afford order {{Order Number}}, cancel it
Expected: cancel_order
TF-IDF: track_order
Embedding: cancel_order

Ticket: I am trying to cancel purchase {{Order Number}}
Expected: cancel_order
TF-IDF: change_order
Embedding: cancel_order

Ticket: I have got to cancel purchase {{Order Number}}
Expected: cancel_order
TF-IDF: change_order
Embedding: cancel_order

Ticket: i need help canceling purchase {{Order Number}}
Expected: cancel_order
TF-IDF: change_order
Embedding:

In [32]:
results = df[["instruction", "intent"]].copy()

results["tfidf_prediction"] = tfidf_pred
results["embedding_prediction"] = embedding_pred

better_cases = results[
    (results["tfidf_prediction"] != results["intent"]) &
    (results["embedding_prediction"] == results["intent"])
]

better_cases.head(10)

,instruction,intent,tfidf_prediction,embedding_prediction
0,question about cancelling order {{Order Number}},cancel_order,track_order,cancel_order
1,i have a question about cancelling oorder {{Or...,cancel_order,track_order,cancel_order
2,i need help cancelling puchase {{Order Number}},cancel_order,track_order,cancel_order
5,can you help me cancel order {{Order Number}}?,cancel_order,track_order,cancel_order
6,"I can no longer afford order {{Order Number}},...",cancel_order,track_order,cancel_order
7,I am trying to cancel purchase {{Order Number}},cancel_order,change_order,cancel_order
8,I have got to cancel purchase {{Order Number}},cancel_order,change_order,cancel_order
9,i need help canceling purchase {{Order Number}},cancel_order,change_order,cancel_order
10,i dont know what to do to cancel order {{Order...,cancel_order,track_order,cancel_order
11,I have a problem with cancelling purchase {{Or...,cancel_order,change_order,cancel_order


In [33]:
import faiss
import numpy as np

# Convert embeddings to float32
kb_vectors = np.array(kb_embeddings).astype("float32")

# Normalize vectors for cosine similarity
faiss.normalize_L2(kb_vectors)

# Create FAISS index
index = faiss.IndexFlatIP(kb_vectors.shape[1])

# Add KB vectors
index.add(kb_vectors)

print("KB vectors:", index.ntotal)

KB vectors: 27


In [34]:
def search_embeddings(query, top_k=5):
    query_vector = model.encode([query]).astype("float32")
    
    faiss.normalize_L2(query_vector)
    
    scores, indices = index.search(query_vector, top_k)

    results = kb.iloc[indices[0]][
        ["article_id", "intent", "category"]
    ].copy()

    results["score"] = scores[0]

    return results.reset_index(drop=True)

In [35]:
query = "I cannot afford purchase {{Order Number}}"

search_embeddings(query)

,article_id,intent,category,score
0,KB_02,change_order,ORDER,0.589627
1,KB_01,cancel_order,ORDER,0.572057
2,KB_16,get_invoice,INVOICE,0.498993
3,KB_26,track_order,ORDER,0.443980
4,KB_20,place_order,ORDER,0.442341


In [36]:
# Encode all tickets
ticket_vectors = model.encode(
    df["instruction"].tolist(),
    show_progress_bar=True
).astype("float32")

# Normalize
faiss.normalize_L2(ticket_vectors)

# Search top 5
scores, indices = index.search(ticket_vectors, 5)

print(scores.shape)
print(indices.shape)

Batches:   0%|          | 0/840 [00:00<?, ?it/s]

(26872, 5)
(26872, 5)


In [37]:
correct = 0

for i in range(len(df)):
    predicted_intent = kb.iloc[indices[i][0]]["intent"]

    if predicted_intent == df.iloc[i]["intent"]:
        correct += 1

faiss_accuracy = correct / len(df)

print(f"FAISS Top-1 Accuracy: {faiss_accuracy:.4f}")

FAISS Top-1 Accuracy: 0.8078


In [38]:
correct = 0

for i in range(len(df)):
    predicted_intents = kb.iloc[indices[i]]["intent"].values

    if df.iloc[i]["intent"] in predicted_intents:
        correct += 1

faiss_recall5 = correct / len(df)

print(f"FAISS Recall@5: {faiss_recall5:.4f}")

FAISS Recall@5: 0.9903


In [39]:
results = df[["instruction", "intent"]].copy()

results["prediction"] = [
    kb.iloc[i]["intent"]
    for i in indices[:, 0]
]

results["correct"] = (
    results["intent"] == results["prediction"]
)

results.head()

,instruction,intent,prediction,correct
0,question about cancelling order {{Order Number}},cancel_order,cancel_order,True
1,i have a question about cancelling oorder {{Or...,cancel_order,cancel_order,True
2,i need help cancelling puchase {{Order Number}},cancel_order,cancel_order,True
3,I need to cancel purchase {{Order Number}},cancel_order,cancel_order,True
4,"I cannot afford this order, cancel purchase {{...",cancel_order,cancel_order,True


In [40]:
error_summary = (
    results.groupby("intent")
    .agg(
        total=("correct", "size"),
        correct=("correct", "sum")
    )
)

error_summary["accuracy"] = (
    error_summary["correct"] / error_summary["total"]
)

error_summary.sort_values("accuracy")

,total,correct,accuracy
intent,,,
get_refund,997,378,0.379137
switch_account,1000,433,0.433000
get_invoice,999,450,0.450450
contact_customer_service,1000,567,0.567000
change_shipping_address,973,560,0.575540
check_refund_policy,997,622,0.623872
delivery_options,995,655,0.658291
delivery_period,999,667,0.667668
track_refund,998,697,0.698397


In [41]:
errors = results[
    results["correct"] == False
]   

errors.head(20)

,instruction,intent,prediction,correct
24,help cnceling order {{Order Number}},cancel_order,change_order,False
47,cance order {{Order Number}},cancel_order,change_order,False
54,I can't afford purchase {{Order Number}},cancel_order,change_order,False
65,I want to cancle purchase {{Order Number}},cancel_order,change_order,False
74,I need help cacneling order {{Order Number}},cancel_order,change_order,False
76,i bought some damn item help to cancep order {...,cancel_order,change_order,False
104,I can't afford purchase {{Order Number}},cancel_order,change_order,False
118,cance order {{Order Number}},cancel_order,change_order,False
120,how could icancel purchase {{Order Number}},cancel_order,change_order,False
126,i no longer want purchase {{Order Number}},cancel_order,change_order,False


In [42]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(df["intent"].unique())

cm = confusion_matrix(
    df["intent"],
    results["prediction"],
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

cm_df

,cancel_order,change_order,change_shipping_address,check_cancellation_fee,check_invoice,check_payment_methods,check_refund_policy,complaint,contact_customer_service,contact_human_agent,...,newsletter_subscription,payment_issue,place_order,recover_password,registration_problems,review,set_up_shipping_address,switch_account,track_order,track_refund
cancel_order,906,91,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
change_order,0,984,0,0,0,0,0,1,0,0,...,0,0,11,0,0,0,0,0,0,0
change_shipping_address,0,0,560,0,0,0,0,0,0,0,...,0,0,0,0,0,0,408,0,0,0
check_cancellation_fee,8,0,0,892,10,0,37,0,1,0,...,0,0,0,0,0,0,0,0,0,0
check_invoice,1,0,0,0,990,0,0,0,1,1,...,0,0,0,0,0,1,0,4,0,0
check_payment_methods,0,0,0,0,79,778,0,1,2,0,...,0,126,6,0,0,1,0,1,0,5
check_refund_policy,0,0,0,0,3,0,622,0,0,0,...,0,0,0,0,0,0,0,0,2,360
complaint,0,0,0,1,3,0,6,973,0,1,...,0,0,0,0,1,2,0,0,0,9
contact_customer_service,49,0,1,0,11,0,1,268,567,46,...,1,2,5,0,44,5,0,0,0,0
contact_human_agent,0,0,0,0,0,0,0,1,9,947,...,0,0,0,0,39,0,0,1,0,1


In [43]:
confusions = (
    results[results["correct"] == False]
    .groupby(["intent", "prediction"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

confusions.head(15)

,intent,prediction,count
94,get_refund,track_refund,545
86,get_invoice,check_invoice,542
7,change_shipping_address,set_up_shipping_address,408
32,check_refund_policy,track_refund,360
77,delivery_period,track_order,325
47,contact_customer_service,complaint,268
154,switch_account,edit_account,226
70,delivery_options,delivery_period,222
163,track_refund,get_refund,203
83,edit_account,switch_account,167


In [44]:
from sklearn.preprocessing import MinMaxScaler

tfidf_scores = similarities
embedding_scores = embedding_similarities

scaler = MinMaxScaler()

tfidf_norm = scaler.fit_transform(tfidf_scores)
embedding_norm = scaler.fit_transform(embedding_scores)

In [45]:
alpha = 0.5

hybrid_scores = (
    alpha * embedding_norm +
    (1 - alpha) * tfidf_norm
)

In [46]:
top_indices = hybrid_scores.argmax(axis=1)

hybrid_pred = kb.iloc[
    top_indices
]["intent"].values

hybrid_accuracy = (
    hybrid_pred == df["intent"].values
).mean()

print(f"Hybrid Top-1 Accuracy: {hybrid_accuracy:.4f}")

Hybrid Top-1 Accuracy: 0.8534


In [47]:
top5 = hybrid_scores.argsort(axis=1)[:, -5:]

correct = 0

for i in range(len(df)):
    predicted_intents = kb.iloc[top5[i]]["intent"].values

    if df.iloc[i]["intent"] in predicted_intents:
        correct += 1

hybrid_recall5 = correct / len(df)

print(f"Hybrid Recall@5: {hybrid_recall5:.4f}")

Hybrid Recall@5: 0.9987


In [48]:
import numpy as np

alphas = np.arange(0, 1.1, 0.1)

results_alpha = []

for alpha in alphas:

    hybrid_scores = (
        alpha * embedding_norm +
        (1 - alpha) * tfidf_norm
    )

    top_indices = hybrid_scores.argmax(axis=1)

    predictions = kb.iloc[
        top_indices
    ]["intent"].values

    accuracy = (
        predictions == df["intent"].values
    ).mean()

    top5 = hybrid_scores.argsort(axis=1)[:, -5:]

    correct = 0

    for i in range(len(df)):
        predicted_intents = kb.iloc[
            top5[i]
        ]["intent"].values

        if df.iloc[i]["intent"] in predicted_intents:
            correct += 1

    recall5 = correct / len(df)

    results_alpha.append([
        alpha,
        accuracy,
        recall5
    ])

alpha_results = pd.DataFrame(
    results_alpha,
    columns=["alpha", "top1_accuracy", "recall_at_5"]
)

alpha_results

,alpha,top1_accuracy,recall_at_5
0,0.0,0.743562,0.970304
1,0.1,0.777910,0.987906
2,0.2,0.804071,0.994381
3,0.3,0.824911,0.997246
4,0.4,0.841322,0.998511
5,0.5,0.853379,0.998698
6,0.6,0.861417,0.998698
7,0.7,0.862496,0.998288
8,0.8,0.854570,0.997283
9,0.9,0.840764,0.994492


In [49]:
alpha_results.sort_values(
    "top1_accuracy",
    ascending=False
)

,alpha,top1_accuracy,recall_at_5
7,0.7,0.862496,0.998288
6,0.6,0.861417,0.998698
8,0.8,0.854570,0.997283
5,0.5,0.853379,0.998698
4,0.4,0.841322,0.998511
9,0.9,0.840764,0.994492
3,0.3,0.824911,0.997246
10,1.0,0.823013,0.987236
2,0.2,0.804071,0.994381
1,0.1,0.777910,0.987906
